# LAB 1 – Bài 6, Trang 45: Groceries Dataset

**Sinh viên:** Lê Nguyễn Anh Dự – **MSSV:** 6451071011

Mỗi dòng là một giao dịch. Ngưỡng support khảo sát: `0.01–0.10` (1%–10%), theo xác nhận; tăng từng 1 điểm phần trăm. Đo thời gian khai phá độc lập cho Apriori và FP-Growth ở từng ngưỡng.


In [ ]:
# Cài thư viện cần thiết và vá jupyter_client/session.py trong runtime hiện tại.
import importlib.util, subprocess, sys, pathlib, re
packages = {'mlxtend':'mlxtend', 'gdown':'gdown', 'pandas':'pandas', 'matplotlib':'matplotlib', 'jupyter_client':'jupyter_client'}
missing = [pkg for mod,pkg in packages.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
spec = importlib.util.find_spec('jupyter_client')
path = pathlib.Path(next(iter(spec.submodule_search_locations))) / 'session.py'
original = path.read_text(encoding='utf-8')
# Dùng timezone-aware UTC; giữ alias utc có sẵn của jupyter_client.
pattern = r'datetime\.utcnow\(\)\.replace\(\s*tzinfo\s*=\s*utc\s*\)'
new, count = re.subn(pattern, 'datetime.now(utc)', original)
if count:
    backup = path.with_suffix('.py.bak_before_utc_patch')
    if not backup.exists(): backup.write_text(original, encoding='utf-8')
    compile(new, str(path), 'exec')
    path.write_text(new, encoding='utf-8')
    print(f'Đã vá {count} lời gọi trong {path}: datetime.now(utc)')
    print('Chọn Runtime > Restart session để tiến trình Python nạp mã mới.')
else:
    print(f'Không có mẫu utcnow() cũ cần thay trong {path}.')
# Chỉ vá runtime hiện tại, không thể vá máy chủ Colab nằm ngoài runtime.


In [ ]:
import csv
from pathlib import Path
from time import perf_counter
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
STUDENT_INFO="Thông tin sinh viên: Lê Nguyễn Anh Dự - 6451071011"
import gdown
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori,fpgrowth
paths=[Path("Dataset/groceries.csv"),Path("../Dataset/groceries.csv"),Path("groceries.csv")]
path=next((p for p in paths if p.exists()),None)
if path is None:
    path=Path("groceries.csv")
    print(f"Đang tải groceries.csv | {STUDENT_INFO}")
    gdown.download(id="1UIeIAeGPrmXaSsFOtfhw9h5gK4RPkseI",output=str(path),quiet=False)
if not path.exists(): raise FileNotFoundError("Không tìm thấy groceries.csv; kiểm tra quyền Drive hoặc tải tệp lên.")
with path.open(encoding="utf-8-sig",newline="") as f: transactions=[[v.strip() for v in row if v.strip()] for row in csv.reader(f)]
transactions=[row for row in transactions if row]
te=TransactionEncoder()
onehot=pd.DataFrame(te.fit(transactions).transform(transactions),columns=te.columns_,dtype=bool)
print(f"Dữ liệu: {len(transactions):,} giao dịch, {onehot.shape[1]} items | {STUDENT_INFO}")
print(f"Ví dụ 5 giỏ hàng | {STUDENT_INFO}"); display(pd.DataFrame({"Transaction":range(1,6),"Items":transactions[:5]}))


## Câu 6a. Apriori – tập phổ biến và thời gian xử lý


In [ ]:
SUPPORTS=[i/100 for i in range(1,11)]
apriori_sets={}; apriori_rows=[]
for sup in SUPPORTS:
    t0=perf_counter(); result=apriori(onehot,min_support=sup,use_colnames=True); seconds=perf_counter()-t0
    result=result.sort_values("support",ascending=False,kind="stable").reset_index(drop=True)
    apriori_sets[sup]=result
    apriori_rows.append({"min_sup":sup,"Số tập phổ biến":len(result),"Thời gian (giây)":seconds})
apriori_time=pd.DataFrame(apriori_rows)
print(f"Apriori: thời gian và số tập phổ biến theo min_sup 1%–10% | {STUDENT_INFO}"); display(apriori_time)


In [ ]:
for sup,result in apriori_sets.items():
    print(f"Apriori – min_sup={sup:.0%}: {len(result)} tập phổ biến | {STUDENT_INFO}")
    display(result)


## Câu 6b. FP-Growth – tập phổ biến và thời gian xử lý


In [ ]:
fp_sets={}; fp_rows=[]
for sup in SUPPORTS:
    t0=perf_counter(); result=fpgrowth(onehot,min_support=sup,use_colnames=True); seconds=perf_counter()-t0
    result=result.sort_values("support",ascending=False,kind="stable").reset_index(drop=True)
    fp_sets[sup]=result
    fp_rows.append({"min_sup":sup,"Số tập phổ biến":len(result),"Thời gian (giây)":seconds})
fp_time=pd.DataFrame(fp_rows)
print(f"FP-Growth: thời gian và số tập phổ biến theo min_sup 1%–10% | {STUDENT_INFO}"); display(fp_time)


In [ ]:
for sup,result in fp_sets.items():
    print(f"FP-Growth – min_sup={sup:.0%}: {len(result)} tập phổ biến | {STUDENT_INFO}")
    display(result)


## Câu 6c. Biểu đồ so sánh và nhận xét


In [ ]:
timing=apriori_time[["min_sup","Thời gian (giây)"]].merge(fp_time[["min_sup","Thời gian (giây)"]],on="min_sup",suffixes=(" – Apriori"," – FP-Growth"))
fig,ax=plt.subplots(figsize=(9,5))
ax.plot(timing.min_sup*100,timing["Thời gian (giây) – Apriori"],marker="o",label="Apriori")
ax.plot(timing.min_sup*100,timing["Thời gian (giây) – FP-Growth"],marker="s",label="FP-Growth")
ax.set(title="Thời gian khai phá theo min_sup",xlabel="min_sup (%)",ylabel="Thời gian (giây)")
ax.set_xticks(timing.min_sup*100); ax.grid(True,alpha=.3); ax.legend(); plt.tight_layout()
print(f"Biểu đồ thời gian Apriori và FP-Growth | {STUDENT_INFO}"); plt.show()
timing["Nhanh hơn"]=timing.apply(lambda r:"Apriori" if r["Thời gian (giây) – Apriori"]<r["Thời gian (giây) – FP-Growth"] else "FP-Growth",axis=1)
print(f"Bảng so sánh thời gian theo support | {STUDENT_INFO}"); display(timing)
print(f"Nhận xét: ngưỡng thấp thường làm số tập phổ biến và thời gian tăng. FP-Growth tránh sinh ứng viên tường minh; kết quả tốc độ còn phụ thuộc dữ liệu và runtime. Đối chiếu biểu đồ/bảng để kết luận lần chạy này. | {STUDENT_INFO}")
